DEPENDÊNCIAS DO SISTEMA OPERACIONAL

- `Python 3.11.5 (recomendado >= 3.8, < 3.12)`
- `PostgreSQL 12 (>= 9.4, < 14)`


LINK DO TUTORIAL:
- https://docs.taiga.io/setup-production.html#setup-prod-from-source-code

In [ ]:
# INICIO
sudo apt-get update
sudo apt-get install -y build-essential binutils-doc autoconf flex bison libjpeg-dev
sudo apt-get install -y libfreetype6-dev zlib1g-dev libzmq3-dev libgdbm-dev libncurses5-dev
sudo apt-get install -y automake libtool curl git tmux gettext
sudo apt-get install -y nginx
sudo apt-get install -y rabbitmq-server #adicionado do docker-compose.yml

# POSTGRESQL
sudo apt-get install -y postgresql-12 postgresql-contrib-12 postgresql-doc-12 postgresql-server-dev-12 #não precisa se usar container do bd
sudo pg_ctlcluster 12 main start #não precisa se usar container do bd

# PYTHON 3 OBRIGATÓRIO123

sudo apt-get install -y python3 python3-pip python3-dev python3-venv #máquina virtual
sudo apt-get install -y libxml2-dev libxslt-dev
sudo apt-get install -y libssl-dev libffi-dev
sudo apt install python3-distutils


# NODE JS
curl -sL https://deb.nodesource.com/setup_16.x | sudo -E bash -
sudo apt-get install -y nodejs

# TEM ESSA OPÇÃO NO TUTORIAL, MAS LOCALMENTE NÃO EXECUTEI
sudo adduser taiga
sudo adduser taiga sudo
sudo su taiga
cd ~

# CONFIGURAR POSTGRESQL E RABBITMQ #já está no docker
sudo -u postgres createuser taiga --interactive --pwprompt 
sudo -u postgres createdb taiga -O taiga --encoding='utf-8' --locale=en_US.utf8 --template=template0

# RABBIT USER e virtualhost RABBITMQ (taiga-events and async tasks) #já está no docker
sudo rabbitmqctl add_user rabbitmquser rabbitmqpassword
sudo rabbitmqctl add_vhost taiga
sudo rabbitmqctl set_permissions -p taiga rabbitmquser ".*" ".*" ".*"

DEPENDÊNCIAS BACKEND

In [ ]:
### CRIAR UMA MÁQUINA VIRTUAL / USEI O ANACONDA-NAVIGATOR
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip setuptools wheel
python -m pip install --upgrade pip-tools

pip install -r requirements.txt 
pip install -r requirements-devel.txt
pip install -r requirements-tests.txt


In [ ]:
pip install git+https://@github.com/celtab/apps-tadt/taigaio/taiga-contrib-protected.git@stable#egg=taiga-contrib-protected

# Ao rodar o comando acima, observe que o vscode vai solicitar seu usuário e senha
# pois a branch está no nosso repositório privado celtab/apps-tadt/taigaio/...

Edit config.py and configure:

* Taiga secret key: **it’s important** to change it. It must have the same value as the secret key in `taiga-events` and `taiga-protected`
* Taiga urls: configure where Taiga would be served using `TAIGA_URL`, `SITES` and `FORCE_SCRIPT_NAME` (see examples below)
* Connection to PostgreSQL; check `DATABASES` section in the file
* Connection to RabbitMQ for `taiga-events`; check "EVENTS" section in the file
* Connection to RabbitMQ for `taiga-async`; check "TAIGA ASYNC" section in the file
* Credentials for email; check "EMAIL" section in the file
* Enable/disable anonymous telemetry; check "TELEMETRY" section in the file

In [ ]:
# Example to configure Taiga in subdomain:

TAIGA_SCHEME = "https"
TAIGA_DOMAIN = "taiga.mycompany.com"
FORCE_SCRIPT_NAME = ""

# Example to configure Taiga in subpath:
TAIGA_SCHEME = "https"
TAIGA_DOMAIN = "taiga.mycompany.com"
FORCE_SCRIPT_NAME = "/taiga"

In [ ]:
# CONFIGURAR ARQUIVO .env

cp .env.example .env

# SEGUIR INSTRUÇÕES DENTRO DO PRÓPRIO ARQUIVO .env

In [ ]:
# EXECUTEI A PARTE ABAIXO EXTRA DADO O SEGUINTE ERRO: erro ImportError: /lib/x86_64-linux-gnu/libp11-kit.so.0: undefined symbol: ffi_type_pointer, version LIBFFI_BASE_7.0

conda update --all
conda install -c conda-forge libffi 
conda install -c conda-forge p11-kit
pip uninstall psycopg2
pip install psycopg2
export LD_LIBRARY_PATH=$CONDA_PREFIX/lib:$LD_LIBRARY_PATH
source ~/.bashrc
conda activate taiga
pip install django psycopg2
source activate taiga

# EXECUTE O CÓDIGO ACIMA NO CASO DO ERRO LIBFFI_BASE_7.0:
# Execute all migrations to populate the database with basic necessary initial data:
(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py migrate --noinput
# create an administrator with strong password
(taiga) CELERY_ENABLED=False DJANGO_SETTINGS_MODULE=settings.config python manage.py createsuperuser
user taiga
email nome.sobrenome@exemplo.org
senha taiga

pip install requests --upgrade --upgrade-strategy eager


(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py loaddata initial_project_templates
(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py compilemessages
(taiga) DJANGO_SETTINGS_MODULE=settings.config python manage.py collectstatic --noinput

In [ ]:
# ADICIONAR UM PROJETO EXEMPLO (OPCIONAL)
CELERY_ENABLED=False DJANGO_SETTINGS_MODULE=settings.config python manage.py sample_data

In [ ]:
# ENFIM RODAR O PROJETO
DJANGO_SETTINGS_MODULE=settings.config python manage.py runserver